# GNNExplainer on Mutagenicity Dataset

This tutorial demonstrates how to use GNNExplainer to explain graph-level
predictions of a GCN model on the **Mutagenicity** dataset.

Workflow:
1. Load the Mutagenicity dataset (drug compounds classified as mutagen/non-mutagen).
2. Train a GCN for binary classification.
3. Implement the `GNNInterface` to connect the trained GCN to GNNExplainer.
4. Run GNNExplainer to find edge and node importance masks.
5. Visualize explanations as molecular graphs with highlighted substructures.

Based on: Ying et al., GNNExplainer: Generating Explanations for Graph Neural Networks (NeurIPS 2019).

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import time
from sklearn.model_selection import train_test_split
from torch_geometric.loader import DataLoader

from kgcnn_torch.models.gcn import GCNModel
from kgcnn_torch.models.gnnexplain import GNNExplainer, GNNInterface
from kgcnn_torch.training.trainer import fit
from kgcnn_torch.training.scheduler import LinearLearningRateScheduler
from kgcnn_torch.utils.devices import get_device

device = get_device("auto")
print(f"Using device: {device}")

## 1. Load Data

The Mutagenicity dataset contains chemical compounds classified as mutagen (1) or
non-mutagen (0). Nodes represent atoms with 14-element one-hot encoding, and edges
represent chemical bonds.

In [ ]:
from kgcnn_torch.data.datasets.MutagenicityDataset import MutagenicityDataset

dataset = MutagenicityDataset()
print(f"Mutagenicity: {len(dataset)} graphs")
print(f"Example graph: {dataset[0]}")

# Element labels corresponding to node features
element_labels = ['C', 'O', 'Cl', 'H', 'N', 'F', 'Br', 'S', 'P', 'I', 'Na', 'K', 'Li', 'Ca']

In [ ]:
# Train/test split
indices = np.arange(len(dataset))
train_idx, test_idx = train_test_split(indices, train_size=0.8, random_state=1)

train_dataset = dataset[torch.tensor(train_idx).long()]
test_dataset = dataset[torch.tensor(test_idx).long()]

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")

## 2. Train a GCN Model

In [ ]:
# Check the node feature dimension from the dataset
sample = dataset[0]
node_input_dim = sample.x.shape[-1] if hasattr(sample, 'x') and sample.x is not None else 14
print(f"Node input dimension: {node_input_dim}")

model = GCNModel(
    node_dim=64,
    depth=3,
    gcn_units=64,
    gcn_activation="relu",
    gcn_pooling="mean",
    node_pooling="sum",
    output_units=[140, 70],
    output_activation="relu",
    output_final_activation="sigmoid",
    num_targets=1,
    output_embedding="graph",
    use_node_embedding=False,   # Mutagenicity uses float node features
    node_input_dim=node_input_dim,
)
model = model.to(device)
print(model)

In [ ]:
# Training
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = LinearLearningRateScheduler(
    optimizer, learning_rate_start=1e-3, learning_rate_stop=1e-4,
    epo_min=100, epo=150
)

history = fit(
    model,
    train_loader=train_loader,
    val_loader=test_loader,
    optimizer=optimizer,
    loss_fn=nn.BCELoss(),
    scheduler=scheduler,
    epochs=150,
    device=device,
    verbose=1,
)

# Plot training curve
plt.figure(figsize=(8, 5))
plt.plot(history['train_loss'], label='Train Loss', color='blue')
plt.plot(history['val_loss'], label='Val Loss', color='red')
plt.xlabel('Epochs')
plt.ylabel('BCE Loss')
plt.title('GCN Training on Mutagenicity')
plt.legend()
plt.show()

## 3. Implement the GNNInterface for GNNExplainer

The `GNNInterface` is an abstract class that wraps a trained GNN model and
provides the methods needed by GNNExplainer to:
- Get predictions (`predict`)
- Get masked predictions (`masked_predict`)
- Query graph dimensions (`get_number_of_nodes`, etc.)
- Generate explanations (`get_explanation`)
- Visualize explanations (`present_explanation`)

In [ ]:
class ExplainableGCN(GNNInterface):
    """Wraps a trained GCN model for use with GNNExplainer.

    The graph_instance is expected to be a single PyG Data object.
    """

    def __init__(self, gnn_model):
        super().__init__()
        self.gnn_model = gnn_model
        self.gnn_model.eval()

    def predict(self, data, **kwargs):
        """Get the GNN's prediction for this graph."""
        with torch.no_grad():
            return self.gnn_model(data)

    def masked_predict(self, data, edge_mask, feature_mask, node_mask, **kwargs):
        """Get prediction with masks applied to edges, features, and nodes."""
        from torch_geometric.data import Data, Batch

        # Clone data to avoid modifying the original
        if isinstance(data, Batch):
            x = data.x.clone()
            edge_attr = data.edge_attr.clone() if data.edge_attr is not None else None
        else:
            x = data.x.clone()
            edge_attr = data.edge_attr.clone() if data.edge_attr is not None else None

        # Apply feature mask: broadcast (F, 1) -> (1, F) for (N, F) node features
        x = x * feature_mask.T.float()
        # Apply node mask: (N, 1) for (N, F) node features
        x = x * node_mask.float()
        # Apply edge mask
        if edge_attr is not None:
            edge_attr = edge_attr * edge_mask.float()

        # Create a new Data object with masked features
        masked_data = Data(
            x=x,
            edge_index=data.edge_index,
            edge_attr=edge_attr,
            batch=data.batch if hasattr(data, 'batch') and data.batch is not None else torch.zeros(x.size(0), dtype=torch.long, device=x.device),
        )
        if hasattr(data, 'edge_weight'):
            masked_data.edge_weight = data.edge_weight

        return self.gnn_model(masked_data)

    def get_number_of_nodes(self, data):
        return data.x.shape[0] if not hasattr(data, 'batch') or data.batch is None else int((data.batch == 0).sum())

    def get_number_of_node_features(self, data):
        return data.x.shape[-1]

    def get_number_of_edges(self, data):
        return data.edge_index.shape[1] if not hasattr(data, 'batch') or data.batch is None else data.edge_index.shape[1]

    def get_explanation(self, data, edge_mask, feature_mask, node_mask, **kwargs):
        """Convert learned masks into a NetworkX graph explanation."""
        edge_relevance = edge_mask[:, 0].detach().cpu().numpy()
        node_relevance = node_mask[:, 0].detach().cpu().numpy()
        feature_relevance = feature_mask[:, 0].detach().cpu().numpy()
        features = data.x.detach().cpu().numpy()
        edges = data.edge_index.T.detach().cpu().numpy()
        num_nodes = features.shape[0]
        num_edges = edges.shape[0]

        graph = nx.Graph()
        for i in range(num_nodes):
            graph.add_node(i, features=features[i], relevance=float(node_relevance[i]))
        for i in range(num_edges):
            e = edges[i]
            graph.add_edge(int(e[0]), int(e[1]), relevance=float(edge_relevance[i]))
        return graph, feature_relevance

    def present_explanation(self, explanation, threshold=0.5):
        """Visualize the explanation as a molecular graph."""
        graph = explanation[0]
        labels_list = element_labels

        color_map = []
        for (u, v, relevance) in graph.edges.data('relevance'):
            r = min((relevance if relevance is not None else 0.1) + 0.1, 1.0)
            color_map.append((0, 0, 0, r))

        node_color_map = []
        node_labels = {}
        for n, f in graph.nodes.data('features'):
            if f is not None:
                element = int(np.argmax(f))
                r, g, b, a = plt.get_cmap('tab20')(element)
                node_color_map.append((r, g, b, graph.nodes[n].get('relevance', 0.5)))
                if element < len(labels_list):
                    node_labels[n] = labels_list[element]
                else:
                    node_labels[n] = str(element)
            else:
                node_color_map.append((0.5, 0.5, 0.5, 0.5))
                node_labels[n] = '?'

        # Check if we have feature relevance to plot
        if np.all(explanation[1] == 1) or np.allclose(explanation[1], 1.0, atol=0.01):
            nx.draw_kamada_kawai(graph, edge_color=color_map,
                                labels=node_labels, node_color=node_color_map)
        else:
            f, axs = plt.subplots(2, figsize=(8, 12))
            nx.draw_kamada_kawai(graph, ax=axs[0], edge_color=color_map,
                                labels=node_labels, node_color=node_color_map)
            bar_colors = [plt.get_cmap('tab20')(i) for i in range(len(labels_list))]
            n_bars = min(len(labels_list), len(explanation[1]))
            axs[1].bar(np.array(labels_list[:n_bars]),
                       explanation[1][:n_bars], color=bar_colors[:n_bars])
            axs[1].set_ylabel('Feature Importance')
            axs[1].set_title('Feature Mask')

## 4. Create GNNExplainer and Explain a Prediction

In [ ]:
# Create the explainable wrapper
explainable_gcn = ExplainableGCN(model)

# Create GNNExplainer with optimizer options
explainer = GNNExplainer(
    explainable_gcn,
    optimizer_options={
        'edge_mask_loss_weight': 0.001,    # Encourage sparse edge mask
        'edge_mask_norm_ord': 1,            # L1 regularization
        'feature_mask_loss_weight': 0,      # No feature mask optimization
        'feature_mask_norm_ord': 1,
        'node_mask_loss_weight': 0,         # No node mask optimization
        'node_mask_norm_ord': 1,
    },
    lr=0.2,
    epochs=100,
    loss_fn='cross_entropy',
)

In [ ]:
# Select a test graph to explain
graph_idx = 42
test_graph = test_dataset[graph_idx]
print(f"Test graph {graph_idx}: {test_graph}")
print(f"True label: {test_graph.y.item()}")

# Move to device for prediction
from torch_geometric.data import Batch
single_batch = Batch.from_data_list([test_graph]).to(device)

# Get the model's prediction
with torch.no_grad():
    pred = model(single_batch)
    print(f"Model prediction: {pred.item():.4f}")
    print(f"Predicted class: {'mutagen' if pred.item() > 0.5 else 'non-mutagen'}")

In [ ]:
# Explain the prediction
inspection_result = explainer.explain(
    single_batch,
    inspection=True,
    verbose=True,
    device=device,
)

In [ ]:
# Visualize the explanation
plt.figure(figsize=(10, 8))
explanation = explainer.get_explanation()
explainer.present_explanation(explanation, threshold=0.5)
plt.title(f"GNNExplainer: Graph {graph_idx} (true={test_graph.y.item()})")
plt.show()

## 5. Inspect the Explanation Process

When `inspection=True`, GNNExplainer returns a dictionary with per-epoch
loss values and predictions, which can be used to verify convergence.

In [ ]:
# Plot predictions over optimization iterations
plt.figure(figsize=(8, 4))
preds = np.array(inspection_result['predictions']).squeeze()
plt.plot(preds)
plt.xlabel('Iterations')
plt.ylabel('GNN output')
plt.title('GNN Prediction During Mask Optimization')
plt.show()

In [ ]:
# Plot total loss
plt.figure(figsize=(8, 4))
plt.plot(inspection_result['total_loss'], label='Total Loss')
plt.plot(inspection_result['pred_loss'], label='Prediction Loss')
plt.plot(inspection_result['reg_loss'], label='Regularization Loss')
plt.xlabel('Iterations')
plt.ylabel('Loss')
plt.title('GNNExplainer Loss Curves')
plt.legend()
plt.show()

In [ ]:
# Plot edge mask loss (sparsity regularization)
if inspection_result['edge_mask_loss']:
    plt.figure(figsize=(8, 4))
    plt.plot(inspection_result['edge_mask_loss'])
    plt.xlabel('Iterations')
    plt.ylabel('Edge Mask Reg. Loss')
    plt.title('Edge Mask Sparsity Regularization')
    plt.show()

## 6. Inspect Learned Masks Directly

In [ ]:
# Get the raw learned masks
masks = explainer.get_masks()
print(f"Edge mask shape: {masks['edge'].shape}")
print(f"Feature mask shape: {masks['feature'].shape}")
print(f"Node mask shape: {masks['node'].shape}")

# Plot edge mask distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(masks['edge'].cpu().numpy().flatten(), bins=50, color='steelblue')
axes[0].set_title('Edge Mask Distribution')
axes[0].set_xlabel('Mask Value (sigmoid-activated)')
axes[0].set_ylabel('Count')

axes[1].hist(masks['node'].cpu().numpy().flatten(), bins=50, color='orange')
axes[1].set_title('Node Mask Distribution')
axes[1].set_xlabel('Mask Value')

axes[2].hist(masks['feature'].cpu().numpy().flatten(), bins=50, color='green')
axes[2].set_title('Feature Mask Distribution')
axes[2].set_xlabel('Mask Value')

plt.tight_layout()
plt.show()

## 7. Explain Multiple Graphs

In [ ]:
# Explain several test graphs
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

explain_indices = [0, 10, 20, 30, 40, 50]

for ax_idx, graph_idx in enumerate(explain_indices):
    if graph_idx >= len(test_dataset):
        break

    test_graph = test_dataset[graph_idx]
    single_batch = Batch.from_data_list([test_graph]).to(device)

    # Get prediction
    with torch.no_grad():
        pred = model(single_batch).item()

    # Explain
    explainer_i = GNNExplainer(
        explainable_gcn,
        optimizer_options={
            'edge_mask_loss_weight': 0.001,
            'edge_mask_norm_ord': 1,
            'feature_mask_loss_weight': 0,
            'feature_mask_norm_ord': 1,
            'node_mask_loss_weight': 0,
            'node_mask_norm_ord': 1,
        },
        lr=0.2,
        epochs=100,
        loss_fn='cross_entropy',
    )
    explainer_i.explain(single_batch, device=device)
    explanation = explainer_i.get_explanation()

    # Visualize
    plt.sca(axes[ax_idx])
    graph = explanation[0]
    color_map = []
    for (u, v, rel) in graph.edges.data('relevance'):
        r = min((rel if rel is not None else 0.1) + 0.1, 1.0)
        color_map.append((0, 0, 0, r))
    node_colors = []
    node_labels = {}
    for n, f in graph.nodes.data('features'):
        if f is not None:
            elem = int(np.argmax(f))
            r, g, b, a = plt.get_cmap('tab20')(elem)
            node_colors.append((r, g, b, graph.nodes[n].get('relevance', 0.5)))
            node_labels[n] = element_labels[elem] if elem < len(element_labels) else '?'
        else:
            node_colors.append((0.5, 0.5, 0.5, 0.5))
            node_labels[n] = '?'
    nx.draw_kamada_kawai(graph, ax=axes[ax_idx], edge_color=color_map,
                         labels=node_labels, node_color=node_colors, node_size=200, font_size=8)
    label_str = 'mutagen' if test_graph.y.item() > 0.5 else 'non-mutagen'
    pred_str = 'mutagen' if pred > 0.5 else 'non-mutagen'
    axes[ax_idx].set_title(f"#{graph_idx}: true={label_str}, pred={pred_str} ({pred:.2f})", fontsize=9)

plt.tight_layout()
plt.suptitle('GNNExplainer Explanations on Mutagenicity', fontsize=14, y=1.02)
plt.show()

## Summary

GNNExplainer workflow in kgcnn-torch:

```python
from kgcnn_torch.models.gnnexplain import GNNExplainer, GNNInterface

# 1. Implement GNNInterface (wrapper around your trained model)
class MyExplainableGNN(GNNInterface):
    def predict(self, data): ...
    def masked_predict(self, data, edge_mask, feature_mask, node_mask): ...
    def get_number_of_nodes(self, data): ...
    def get_number_of_edges(self, data): ...
    def get_number_of_node_features(self, data): ...
    def get_explanation(self, data, edge_mask, feature_mask, node_mask): ...
    def present_explanation(self, explanation): ...

# 2. Create GNNExplainer
explainer = GNNExplainer(
    MyExplainableGNN(trained_model),
    optimizer_options={...},
    lr=0.2, epochs=100,
)

# 3. Explain a graph
info = explainer.explain(graph_data, inspection=True)

# 4. Get and visualize explanation
explanation = explainer.get_explanation()
explainer.present_explanation(explanation)
```

## 8. Weighted Model for Better Node Explanations

Instead of masking node features directly (multiplying by node_mask), we use
a **weighted pooling** approach: `GCNWeightedModel` accepts per-node weights
that scale each node's contribution during the graph-level readout.

This is the approach used in the Keras `mutagenicity_2` tutorial. The key idea:
- Train a standard GCN model (already done above).
- Create a `GCNWeightedModel` with the same architecture.
- Transfer the trained weights (both models share the same parameters, only the pooling layer differs).
- In `masked_predict`, pass `node_mask` as the `node_weight` argument for weighted pooling.

In [ ]:
# Atomic number -> element index mapping for the Mutagenicity dataset
# The dataset stores raw atomic numbers in data.x (shape N,1)
node_translate = np.array([6, 8, 17, 1, 7, 9, 35, 16, 15, 53, 11, 19, 3, 20])
z_to_element_idx = {int(z): i for i, z in enumerate(node_translate)}

from kgcnn_torch.models.gcn import GCNWeightedModel

# Create a weighted model with the same architecture as the trained model
weighted_model = GCNWeightedModel(
    node_dim=64,
    depth=3,
    gcn_units=64,
    gcn_activation="relu",
    gcn_pooling="mean",
    node_pooling="sum",
    output_units=[140, 70],
    output_activation="relu",
    output_final_activation="sigmoid",
    num_targets=1,
    output_embedding="graph",
    use_node_embedding=False,
    node_input_dim=node_input_dim,
)

# Transfer weights from the trained GCNModel.
# Both models have identical parameters (node_projection, dense_in, convs, output_mlp).
# Only the pooling layer differs (PoolingNodes vs PoolingWeightedNodes), and neither has
# learnable parameters, so load_state_dict works directly.
weighted_model.load_state_dict(model.state_dict())
weighted_model = weighted_model.to(device)
weighted_model.eval()
print("Weights transferred successfully from GCNModel -> GCNWeightedModel")

In [ ]:
class ExplainableGCNWeighted(GNNInterface):
    """Wraps a GCNWeightedModel for GNNExplainer.

    Key difference from ExplainableGCN: node_mask is passed as node_weight
    for weighted graph pooling, rather than multiplying node features directly.
    """

    def __init__(self, gnn_model):
        super().__init__()
        self.gnn_model = gnn_model
        self.gnn_model.eval()

    def predict(self, data, **kwargs):
        from torch_geometric.data import Data
        with torch.no_grad():
            pred_data = Data(
                x=data.x,
                edge_index=data.edge_index,
                edge_attr=data.edge_attr if hasattr(data, 'edge_attr') and data.edge_attr is not None else None,
                batch=data.batch if hasattr(data, 'batch') and data.batch is not None else torch.zeros(data.x.size(0), dtype=torch.long, device=data.x.device),
                node_weight=torch.ones(data.x.size(0), 1, device=data.x.device),
            )
            if hasattr(data, 'edge_weight'):
                pred_data.edge_weight = data.edge_weight
            return self.gnn_model(pred_data)

    def masked_predict(self, data, edge_mask, feature_mask, node_mask, **kwargs):
        from torch_geometric.data import Data
        x = data.x.clone()
        edge_attr = data.edge_attr.clone() if hasattr(data, 'edge_attr') and data.edge_attr is not None else None

        # Apply feature mask
        x = x * feature_mask.T.float()
        # Apply edge mask
        if edge_attr is not None:
            edge_attr = edge_attr * edge_mask.float()

        # Pass node_mask as node_weight for weighted pooling (already sigmoid-activated)
        masked_data = Data(
            x=x,
            edge_index=data.edge_index,
            edge_attr=edge_attr,
            batch=data.batch if hasattr(data, 'batch') and data.batch is not None else torch.zeros(x.size(0), dtype=torch.long, device=x.device),
            node_weight=node_mask.float(),
        )
        if hasattr(data, 'edge_weight'):
            masked_data.edge_weight = data.edge_weight
        return self.gnn_model(masked_data)

    def get_number_of_nodes(self, data):
        return data.x.shape[0] if not hasattr(data, 'batch') or data.batch is None else int((data.batch == 0).sum())

    def get_number_of_node_features(self, data):
        return data.x.shape[-1]

    def get_number_of_edges(self, data):
        return data.edge_index.shape[1]

    def get_explanation(self, data, edge_mask, feature_mask, node_mask, **kwargs):
        edge_relevance = edge_mask[:, 0].detach().cpu().numpy()
        node_relevance = node_mask[:, 0].detach().cpu().numpy()
        feature_relevance = feature_mask[:, 0].detach().cpu().numpy()
        features = data.x.detach().cpu().numpy()
        edges = data.edge_index.T.detach().cpu().numpy()
        num_nodes = features.shape[0]
        num_edges = edges.shape[0]

        graph = nx.Graph()
        for i in range(num_nodes):
            graph.add_node(i, features=features[i], relevance=float(node_relevance[i]))
        for i in range(num_edges):
            e = edges[i]
            graph.add_edge(int(e[0]), int(e[1]), relevance=float(edge_relevance[i]))
        return graph, feature_relevance

    def present_explanation(self, explanation, threshold=0.5):
        graph = explanation[0]
        color_map = []
        for (u, v, relevance) in graph.edges.data('relevance'):
            r = min((relevance if relevance is not None else 0.1) + 0.1, 1.0)
            color_map.append((0, 0, 0, r))

        node_color_map = []
        node_labels_map = {}
        for n, f in graph.nodes.data('features'):
            if f is not None:
                z = int(round(f[0]))
                idx = z_to_element_idx.get(z, 0)
                r, g, b, a = plt.get_cmap('tab20')(idx)
                node_color_map.append((r, g, b, graph.nodes[n].get('relevance', 0.5)))
                node_labels_map[n] = element_labels[idx] if idx < len(element_labels) else '?'
            else:
                node_color_map.append((0.5, 0.5, 0.5, 0.5))
                node_labels_map[n] = '?'

        if np.all(explanation[1] == 1) or np.allclose(explanation[1], 1.0, atol=0.01):
            nx.draw_kamada_kawai(graph, edge_color=color_map,
                                labels=node_labels_map, node_color=node_color_map)
        else:
            f, axs = plt.subplots(2, figsize=(8, 12))
            nx.draw_kamada_kawai(graph, ax=axs[0], edge_color=color_map,
                                labels=node_labels_map, node_color=node_color_map)
            bar_colors = [plt.get_cmap('tab20')(i) for i in range(len(element_labels))]
            n_bars = min(len(element_labels), len(explanation[1]))
            axs[1].bar(np.array(element_labels[:n_bars]),
                       explanation[1][:n_bars], color=bar_colors[:n_bars])
            axs[1].set_ylabel('Feature Importance')
            axs[1].set_title('Feature Mask')

In [ ]:
# Create explainer using the weighted model
explainable_weighted = ExplainableGCNWeighted(weighted_model)

explainer_w = GNNExplainer(
    explainable_weighted,
    optimizer_options={
        'edge_mask_loss_weight': 0.005,
        'edge_mask_norm_ord': 1,
        'feature_mask_loss_weight': 0.0001,
        'feature_mask_norm_ord': 1,
        'node_mask_loss_weight': 0.0001,   # Non-zero: optimizes node importance via weighted pooling
        'node_mask_norm_ord': 1,
    },
    lr=0.2,
    epochs=100,
    loss_fn='cross_entropy',
)

# Explain a test graph using the weighted model
graph_idx_w = 42
test_graph_w = test_dataset[graph_idx_w]
single_batch_w = Batch.from_data_list([test_graph_w]).to(device)

with torch.no_grad():
    pred_w = explainable_weighted.predict(single_batch_w)
    print(f"Weighted model prediction for graph {graph_idx_w}: {pred_w.item():.4f}")

inspection_w = explainer_w.explain(single_batch_w, inspection=True, verbose=True, device=device)

In [ ]:
# Visualize explanation from weighted model
plt.figure(figsize=(10, 8))
explanation_w = explainer_w.get_explanation()
explainer_w.present_explanation(explanation_w)
plt.title(f"Weighted GNNExplainer: Graph {graph_idx_w} (true={test_graph_w.y.item()})")
plt.show()

# Compare node mask distributions: node masks are now actively optimized
masks_w = explainer_w.get_masks()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(masks_w['edge'].cpu().numpy().flatten(), bins=50, color='steelblue')
axes[0].set_title('Edge Mask (Weighted)')
axes[1].hist(masks_w['node'].cpu().numpy().flatten(), bins=50, color='orange')
axes[1].set_title('Node Mask (Weighted)')
axes[2].hist(masks_w['feature'].cpu().numpy().flatten(), bins=50, color='green')
axes[2].set_title('Feature Mask (Weighted)')
plt.tight_layout()
plt.show()

## 9. Counterfactual Explanation

By default, GNNExplainer explains the model's actual prediction: for a mutagenic
molecule, it highlights substructures responsible for mutagenicity.

We can instead provide `output_to_explain` to create a **counterfactual** explanation.
Setting `output_to_explain=0` tells the explainer: "find substructures that, when
highlighted, push the prediction toward non-mutagenic." This reveals what structures
are most responsible for the mutagenic classification.

In [ ]:
# Find the 100 most mutagenic molecules (highest prediction scores)
all_preds = []
with torch.no_grad():
    for i in range(len(test_dataset)):
        tb = Batch.from_data_list([test_dataset[i]]).to(device)
        p = model(tb).item()
        all_preds.append(p)
all_preds = np.array(all_preds)

# Most mutagenic = highest prediction (close to 1.0)
most_mutagenic_indices = np.argsort(-all_preds)[:100]
print("Top 100 most mutagenic molecules (test set indices):")
print(most_mutagenic_indices[:20], "...")
print(f"Predictions range: {all_preds[most_mutagenic_indices[0]]:.4f} to {all_preds[most_mutagenic_indices[-1]]:.4f}")

In [ ]:
# Select one mutagenic molecule and explain it normally first
instance_index = most_mutagenic_indices[5]
test_graph_cf = test_dataset[instance_index]
single_batch_cf = Batch.from_data_list([test_graph_cf]).to(device)

with torch.no_grad():
    pred_cf = model(single_batch_cf).item()
print(f"Graph {instance_index}: true={test_graph_cf.y.item()}, pred={pred_cf:.4f}")

# Standard explanation (why is this molecule mutagenic?)
explainer_cf = GNNExplainer(
    explainable_weighted,
    optimizer_options={
        'edge_mask_loss_weight': 0.005,
        'edge_mask_norm_ord': 1,
        'feature_mask_loss_weight': 0.0001,
        'feature_mask_norm_ord': 1,
        'node_mask_loss_weight': 0.0001,
        'node_mask_norm_ord': 1,
    },
    lr=0.1,
    epochs=100,
    loss_fn='cross_entropy',
)
explainer_cf.explain(single_batch_cf, device=device)

plt.figure(figsize=(10, 8))
explainer_cf.present_explanation(explainer_cf.get_explanation())
plt.title(f"Standard: Why is graph {instance_index} mutagenic? (pred={pred_cf:.4f})")
plt.show()

In [ ]:
# Counterfactual explanation: explain toward output=0 (non-mutagenic)
# This highlights the substructures that make this molecule mutagenic
# by finding what would need to change for it to be non-mutagenic.
explainer_counter = GNNExplainer(
    explainable_weighted,
    optimizer_options={
        'edge_mask_loss_weight': 0.005,
        'edge_mask_norm_ord': 1,
        'feature_mask_loss_weight': 0.0001,
        'feature_mask_norm_ord': 1,
        'node_mask_loss_weight': 0.0001,
        'node_mask_norm_ord': 1,
    },
    lr=0.1,
    epochs=100,
    loss_fn='cross_entropy',
)

explainer_counter.explain(
    single_batch_cf,
    output_to_explain=torch.tensor([0.], device=device),
    device=device,
)

plt.figure(figsize=(10, 8))
explainer_counter.present_explanation(explainer_counter.get_explanation())
plt.title(f"Counterfactual: Why could graph {instance_index} be non-mutagenic?")
plt.show()

## 10. Batch Explanation & Clustering

Following the Keras `mutagenicity_3` tutorial, we:
1. Sample 200 mutagenic molecules from the test set.
2. Generate GNNExplainer explanations for all of them.
3. Convert explanations to fixed-length vectors (bond-type matrices).
4. Apply agglomerative clustering to find common explanation patterns.
5. Visualize the dendrogram and representative explanations per cluster.

In [ ]:
# Sample 200 mutagenic molecules (pred > 0.5)
mutagenic_mask = all_preds > 0.5
mutagenic_indices = np.where(mutagenic_mask)[0]
print(f"Total mutagenic molecules in test set: {len(mutagenic_indices)}")

np.random.seed(42)
n_sample = min(200, len(mutagenic_indices))
sampled_indices = np.random.choice(mutagenic_indices, n_sample, replace=len(mutagenic_indices) < 200)
print(f"Sampled {n_sample} mutagenic molecules for batch explanation")

In [ ]:
# Generate explanations for all sampled molecules
explanations = []
for i, mol_idx in enumerate(sampled_indices):
    test_g = test_dataset[mol_idx]
    sb = Batch.from_data_list([test_g]).to(device)

    exp_i = GNNExplainer(
        explainable_weighted,
        optimizer_options={
            'edge_mask_loss_weight': 0.001,
            'edge_mask_norm_ord': 1,
            'feature_mask_loss_weight': 0,
            'feature_mask_norm_ord': 1,
            'node_mask_loss_weight': 0,
            'node_mask_norm_ord': 1,
        },
        lr=0.2,
        epochs=100,
        loss_fn='cross_entropy',
    )
    exp_i.explain(sb, device=device)
    explanations.append(exp_i.get_explanation())
    if (i + 1) % 20 == 0:
        print(f"  Explained {i + 1}/{n_sample}")

print(f"Generated {len(explanations)} explanations")

In [ ]:
def explanation_to_vector(explanation):
    """Convert an explanation graph to a fixed-length bond-type vector.

    Creates a 14x14 bond matrix where entry (i, j) sums edge relevances
    between element type i and element type j. Returns the upper-triangular
    part as a flat vector, normalized to sum to 1.
    """
    graph = explanation[0]
    bond_matrix = np.zeros((14, 14))
    for (u, v, relevance) in graph.edges.data('relevance'):
        f_u = graph.nodes[u].get('features')
        f_v = graph.nodes[v].get('features')
        if f_u is None or f_v is None:
            continue
        # Map atomic number to element index
        z_u = int(round(f_u[0]))
        z_v = int(round(f_v[0]))
        idx_u = z_to_element_idx.get(z_u, 0)
        idx_v = z_to_element_idx.get(z_v, 0)
        rel = relevance if relevance is not None else 0.0
        bond_matrix[idx_u, idx_v] += rel
        bond_matrix[idx_v, idx_u] += rel
    # Extract upper triangular as flat vector
    bond_vector = bond_matrix[np.triu_indices(14)]
    total = np.sum(bond_vector)
    if total > 0:
        bond_vector = bond_vector / total
    return bond_vector

explanation_vectors = [explanation_to_vector(expl) for expl in explanations]
print(f"Explanation vectors: {len(explanation_vectors)} vectors of dim {len(explanation_vectors[0])}")

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import cdist
from sklearn.cluster import AgglomerativeClustering

# Dendrogram of explanation vectors
plt.figure(figsize=(14, 6))
linked = linkage(explanation_vectors, 'complete', metric='cityblock')
dendrogram(linked,
           orientation='top',
           distance_sort='descending',
           show_leaf_counts=True)
plt.title('Dendrogram of GNNExplainer Explanation Vectors')
plt.xlabel('Sample Index')
plt.ylabel('Manhattan Distance')
plt.show()

In [ ]:
# Agglomerative clustering
num_clusters = 7
clustering = AgglomerativeClustering(
    n_clusters=num_clusters, metric='manhattan', linkage='complete'
).fit(explanation_vectors)

# Print cluster sizes
for c in range(num_clusters):
    count = np.sum(clustering.labels_ == c)
    print(f"Cluster {c}: {count} molecules")

In [ ]:
# Show representative explanation for each cluster
# (the explanation closest to the cluster centroid)
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for cluster_idx in range(num_clusters):
    cluster_mask = clustering.labels_ == cluster_idx
    cluster_vectors = np.array([explanation_vectors[i] for i in np.where(cluster_mask)[0]])
    cluster_explanations = [explanations[i] for i in np.where(cluster_mask)[0]]

    # Find the explanation closest to the cluster centroid
    centroid = np.mean(cluster_vectors, axis=0)
    dists = cdist(np.array([centroid]), cluster_vectors, metric='cityblock')[0]
    representative_idx = np.argmin(dists)

    plt.sca(axes[cluster_idx])
    expl = cluster_explanations[representative_idx]
    graph = expl[0]

    # Draw the representative explanation
    color_map = []
    for (u, v, rel) in graph.edges.data('relevance'):
        r = min((rel if rel is not None else 0.1) + 0.1, 1.0)
        color_map.append((0, 0, 0, r))
    node_colors = []
    node_labs = {}
    for n, f in graph.nodes.data('features'):
        if f is not None:
            z = int(round(f[0]))
            idx = z_to_element_idx.get(z, 0)
            r, g, b, a = plt.get_cmap('tab20')(idx)
            node_colors.append((r, g, b, graph.nodes[n].get('relevance', 0.5)))
            node_labs[n] = element_labels[idx] if idx < len(element_labels) else '?'
        else:
            node_colors.append((0.5, 0.5, 0.5, 0.5))
            node_labs[n] = '?'
    nx.draw_kamada_kawai(graph, ax=axes[cluster_idx], edge_color=color_map,
                         labels=node_labs, node_color=node_colors, node_size=200, font_size=8)
    axes[cluster_idx].set_title(f"Cluster {cluster_idx} (n={np.sum(cluster_mask)})", fontsize=10)

# Hide unused subplot
if num_clusters < len(axes):
    for j in range(num_clusters, len(axes)):
        axes[j].axis('off')

plt.tight_layout()
plt.suptitle('Representative Explanations per Cluster', fontsize=14, y=1.02)
plt.show()